# 演習1. スレッドとは何か ―― なぜ分担すると速くなるのか

**このノートの進め方**：上のセルから順に ▶ を押していくだけです。
コードセルの1行目にある `%%writefile` は「このセルの中身をファイルとして保存する」という
Colab の命令で、次のセルでそれをコンパイル・実行します。

---

このノートで扱うのは「**なぜスレッドを使うと速くなるのか**」です。
まず言葉の整理から始めます。

## タスク・プロセス・スレッド

まず言葉を整理します。並行処理の話では、次の3つがよく出てきます。

**タスク（task）**：「やるべき仕事」を指す、いちばん広い言葉です。
プログラムの用語というより、日常語に近い使われ方をします。
「動画を1フレーム読む」も「画像から物体を検出する」もタスクです。

**プロセス（process）**：OS がタスクを実行するときの単位です。
プログラムを起動すると、OS はプロセスを1つ作り、そこにメモリを割り当てて実行を始めます。
`./a.out` と打つたびに、プロセスが1つ生まれています。

**スレッド（thread）**：プロセスの中にある「実行の流れ」です。
プロセスが生まれた時点で、`main()` を実行するスレッドが1本入っています。
`std::thread` はこの流れを**増やす**ための道具です。

```
コンピュータ
 ├─ プロセスA（例：ブラウザ）
 │    ├─ スレッド
 │    └─ スレッド
 │
 └─ プロセスB（自分のプログラム）        ← ./a.out で生まれる
      ├─ スレッド ⇒ main() を実行する（最初からある）
      ├─ スレッド ⇒ std::thread で増やした
      └─ スレッド ⇒ std::thread で増やした
```

## なぜ「プロセス」ではなく「スレッド」なのか

仕事を分担させたいなら、プロセスを増やす方法もあります（実際そうする場面もあります）。
それでもスレッドを使うのは、**メモリの扱いが決定的に違う**からです。

**メモリ**

- **プロセス**：基本は各自が専用。共有したければ共有メモリなどの仕組みを**自分で用意**する
- **スレッド**：**何もしなくても全スレッドで共有**

**データの受け渡し**

- **プロセス**：通信・ファイル・共有メモリなど、**仕組みを選んで作り込む**
- **スレッド**：**同じ変数を直接見るだけ**

**作る手間**

- **プロセス**：重い　／　**スレッド**：軽い

**相手を壊せるか**

- **プロセス**：壊しにくい（仕切りがあるので）
- **スレッド**：**簡単に壊せる**（仕切りがないので）

プロセス間でもデータは共有**できます**（共有メモリ、パイプ、ソケットなど、OS が仕組みを
用意しています）。ただし、どれも「その仕組みを呼び出して、使い方を設計する」という
作り込みが必要です。スレッドなら、`std::ref` で同じ変数を渡すだけで済みます。

画像1枚は数百KB〜数MBあります。この規模のデータを頻繁に受け渡すプログラムでは、
「何もしなくても同じメモリが見えている」スレッドの手軽さが効いてきます。

**ただし、この「共有できる」という長所は、そのまま短所でもあります。**
同じ変数に2つのスレッドが同時に触れば、値が壊れます。
だから並行処理の話は、いつも「速くする話」と「壊さない話」がセットになります。

- このノート（演習1） ⇒ **速くする話**。スレッドで分担するとなぜ速くなるのか
- 演習2以降 ⇒ **壊さない話**。共有したものをどう守るのか

まずは、なぜ分担すると速くなるのかを、図と実測で確かめます。

## 分担のしかたは2通りある

スレッドで仕事を分担する形は、大きく分けて2つあります。
**この2つは図の見た目がまったく違う**ので、先に区別しておきます。

### ケース1：仕事どうしが独立 ―― そのまま同時に走らせる（並列）

「Aさんの書類仕事」と「Bさんの書類仕事」のように、**互いに無関係な仕事**なら、
2つのスレッドでそのまま同時に走らせるだけです。
どちらも10分かかる仕事だとして、図にしてみます。

```
【1人で順番に】                          【2スレッドで】
     0    10   20 (分)                        0    10 (分)
  A  AAAAA.....                            A  AAAAA
  B  .....BBBBB                            B  BBBBB
     合計 20分                                 合計 10分
```

（`A` `B` は working、`.` は何もしていない時間）

- **1人で順番にやる場合**：A の区間（0〜10分）と B の区間（10〜20分）は
  **まったく重なりません**。B は A が終わるまで待つしかないので、
  前半はずっと `.` です。だから合計 20分かかります
- **2スレッドの場合**：A も B も **0〜10分の同じ区間**にいます。
  つまり **完全に重なっています**。`.` が消え、合計は 10分になります

このように **行がそろって重なる**形を **並列処理** と呼びます。
**1-1 で確かめるのはこちらです。**

### ケース2：1つの仕事が「段」の連なり ―― ずらして重ねる（パイプライン）

一方、「データを読む → 処理する → 結果を出す」のように、
**1件の中に順序のある段**があると、同じ件の Read と Infer を同時にはできません
（読み終わっていないデータは処理できないから）。

- **Read**：データを1つ取り出す（動画から1フレーム読む）
- **Infer**：重い処理をする（画像から物体を検出する）
- **Show**：結果を出す（画面に表示する）

それでも分担する方法があります。**件をまたいで重ねる**のです。
Read係がデータ2を読んでいる間に、Infer係はデータ1を処理する。

```
1文字 = 5ms、数字はデータ番号、'.' は「何もしていない」（各段 30ms とする）

【1人で順番に】                              【3人で分担（パイプライン）】
Read   111111............222222......        Read   111111222222333333444444
Infer  ......111111............222222        Infer  ......111111222222333333
Show   ............111111............        Show   ............111111222222
```

同じ件の段は**ずれたまま**、別の件どうしが**重なる** ―― この「工場の流れ作業」の形を
**パイプライン** と呼びます。ケース1のように完全には重なりません。
**1-3 で確かめるのはこちらです。**

> **用語メモ**：複数の流れを同時に進行させる考え方全体を**並行処理**と呼び、
> そのうち物理的に同時に実行されているものを**並列処理**と呼びます
> （並行は1コアでも切り替えで実現できますが、並列には複数のコアが必要です）。
> 厳密な区別は今は不要ですが、「図が重なる＝並列」「ずれて重なる＝パイプライン」の
> 2つの絵は区別して覚えてください。

## 速さの物差しは2つある ―― レイテンシとスループット

「速くなった」と言うとき、実は2つの別々の物差しがあります。

- **レイテンシ**（latency）：**1件**が入ってから結果が出るまでの時間
  　例）1フレーム 90ms
- **スループット**（throughput）：単位時間あたりに**何件**さばけるか
  　例）33 FPS ―― **FPS はスループットです**

高速道路にたとえると、レイテンシは「入口から出口までの所要時間」、
スループットは「1時間に何台通過するか」です。
車を増やしても1台の所要時間は縮みませんが、通過台数は増やせます。

スレッドによる分担も同じです。

- **ケース1（並列）**
  - レイテンシ（1件の時間）：**変わらない**（A の仕事は10分のまま）
  - スループット（件/秒）：2人なら**2倍**
- **ケース2（パイプライン）**
  - レイテンシ（1件の時間）：**変わらない**（1件は 30+30+30=90ms のまま）
  - スループット（件/秒）：3人で**最大3倍**

> **スレッドは1件を速くしない。同じ時間に多くの件をさばけるようにする。**
> つまり改善するのは**スループット**であって、レイテンシではありません。

動画処理で上げたい「FPS」はスループットです。だからスレッドが効きます。
逆に「1枚の写真をとにかく早く処理したい」（レイテンシ）なら、
スレッドで分担してもほとんど縮みません。この区別はこの先ずっと使います。

## 1-1. ケース1を確かめる ―― 独立な仕事は完全に重なる

まず「スレッドを起動する」とはどういうことかを見ます。

**仕事A**（300ms かかる）と **仕事B**（300ms かかる）の2つを片づけます。
互いに無関係な仕事なので、順序はどちらが先でも構いません。

- **1スレッドで**：1本のスレッドが 仕事A → 仕事B の順にこなす
- **2スレッドで**：スレッド1が仕事A、スレッド2が仕事B を受け持つ

実行すると、**どのスレッドがいつ働いていたか**がタイムチャートになって出ます。
図では **1行が1本のスレッド**、行の中の `A` `B` が**どちらの仕事をしているか**を表します。

**実行前に予測してください。それぞれ何 ms かかるでしょうか。**

In [ ]:
%%writefile ex01a.cpp
#include <iostream>
#include <string>
#include <algorithm>
#include <thread>
#include <chrono>
using namespace std;
using namespace std::chrono;

const int SCALE = 20;                    // 図の1文字あたりの時間(ms)
steady_clock::time_point t0;
int t_begin[2], t_end[2];                // 仕事A(=0) と 仕事B(=1) の開始/終了時刻
const char MARK[2] = {'A', 'B'};

int now_ms() { return (int)duration_cast<milliseconds>(steady_clock::now() - t0).count(); }

// 300ms かかる仕事を1つこなす（id=0 が仕事A、id=1 が仕事B）
void job(int id) {
    t_begin[id] = now_ms();
    this_thread::sleep_for(milliseconds(300));
    t_end[id] = now_ms();
}
// ※ 実行中は何も表示しない。開始・終了時刻だけを記録し、最後に draw() がまとめて図を描く。
//    （複数のスレッドが同時に画面へ書き込むと表示が乱れることがあるため。詳しくは演習2で）

void draw(bool oneThread) {
    int last = max(t_end[0], t_end[1]);
    int w = last / SCALE;

    if (oneThread) {
        // スレッドは1本。その1本が仕事A→仕事Bの順にこなす
        string s(w, '.');
        for (int id = 0; id < 2; id++)
            for (int c = t_begin[id] / SCALE; c < t_end[id] / SCALE && c < w; c++) s[c] = MARK[id];
        cout << "T1   " << s << "\n";
        cout << "Total " << last << " ms\n";
    } else {
        for (int id = 0; id < 2; id++) {
            string s(w, '.');
            for (int c = t_begin[id] / SCALE; c < t_end[id] / SCALE && c < w; c++) s[c] = MARK[id];
            cout << "T" << (id + 1) << "   " << s << "\n";
        }
        int ov = min(t_end[0], t_end[1]) - max(t_begin[0], t_begin[1]);
        if (ov < 0) ov = 0;
        cout << "Total " << last << " ms   （T1 と T2 が同時に働いた時間 = " << ov << " ms）\n";
    }
}

int main() {
    cout << "図の見かた： 1文字 = 20ms（帯の長さがそのまま所要時間）\n"
         << "  T1 / T2 = スレッド（行が1本 = スレッド1本）\n"
         << "  A = 仕事A（300ms）、B = 仕事B（300ms）、'.' = 何もしていない\n";

    cout << "\n【1スレッドで順番に】   スレッドは1本だけ\n";
    t0 = steady_clock::now();
    job(0);
    job(1);
    draw(true);

    cout << "\n【2スレッドで】   スレッドは2本\n";
    t0 = steady_clock::now();
    thread t1(job, 0);
    thread t2(job, 1);
    t1.join();
    t2.join();
    draw(false);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex01a.cpp -o ex01a && ./ex01a

1人でやると 600ms、2つのスレッドに分けると 300ms。**半分**になりました。

```
【1スレッドで順番に】   スレッドは1本だけ
T1   AAAAAAAAAAAAAAABBBBBBBBBBBBBBB
Total 600 ms

【2スレッドで】   スレッドは2本
T1   AAAAAAAAAAAAAAA
T2   BBBBBBBBBBBBBBB
Total 300 ms   （T1 と T2 が同時に働いた時間 = 300 ms）
```

図の読み方は3つだけです。

- **横が時間**（1文字 = 20ms）。**帯の長さがそのまま所要時間**
- **1行 = 1本のスレッド**（`T1` `T2`）。**行の本数がスレッドの本数**
- 行の中の文字が、そのスレッドが**どちらの仕事をしているか**（`A` = 仕事A、`B` = 仕事B）

読み取れることは次のとおりです。

- **1スレッドの場合**：行は **`T1` の1本だけ**。
  その1本の中に仕事A（`AAA…`）と仕事B（`BBB…`）が**順番に並びます**。
  スレッドが1本しかないので、2つの仕事が同時に進むことはありません。600ms かかります
- **2スレッドの場合**：行が **`T1` と `T2` の2本**になり、
  **どちらも左端（＝0 ms）から始まっています**。
  2本の行が縦にそろっている ―― これが「同時に動いている」ということです。だから 300ms

仕事A と 仕事B は独立なので、遠慮なく同時に走れます。**ケース1（並列）の絵そのもの**です。
パイプライン（ケース2）のように**ずれて**重なるのではありません。
ずれるのは「段に順序がある」ときだけで、ここには順序がないので、まるごと重なります。

レイテンシとスループットで言うと：

- **レイテンシは変わっていません**。仕事A はどちらの場合も 300ms かかっています
- 変わったのは**全体のスループット**です。600ms で2件 → 300ms で2件になりました

**行が1本増えると、その分だけ同時に進む** ―― これがスレッドの効果です。

### 覚えるべき3つ

```cpp
std::thread t1(job, 0);           // ① スレッドを作る（第1引数が関数、あとは その関数への引数）
t1.join();                        // ② そのスレッドが終わるまで待つ
```

```bash
g++ -std=c++17 -pthread ex01a.cpp -o ex01a    # ③ -pthread が必要
```

- **`join()` を忘れると**、`main` が先に終わってしまい、プログラムが異常終了します
- **`-pthread` を忘れると**、ビルドに失敗することがあります
- スレッドに**参照を渡したいときは `std::ref(変数)`** が必要です（C++問題集の問9を参照）

## 1-2. 【予測クイズ】人数を増やせば増やすほど速くなるか

1-1 の仕事（`job()`）は `sleep_for`、つまり「**待つだけ**」の仕事でした。
今度は、性質の違う2種類の仕事を比べます。

- **待つ仕事** ⇒ 300ms 待つだけ（ファイルの読み込みや、外部ハードウェアの応答待ちのモデル）
- **計算する仕事** ⇒ CPU で計算し続ける（およそ300ms分）

それぞれ4件を、1人・2人・4人で分担します。

**どちらも人数に比例して速くなるでしょうか？** 予測してから実行してください。

In [ ]:
%%writefile ex01b.cpp
#include <iostream>
#include <thread>
#include <vector>
#include <chrono>
using namespace std::chrono;

// 待つ仕事：300ms 待つだけ（ファイル読み込みや外部ハードウェアの応答待ちのモデル）
void wait_job(int) { std::this_thread::sleep_for(milliseconds(300)); }

// 計算する仕事：CPU で計算し続ける（およそ300ms分）
long calc_job(int seed) {
    long s = 0;
    for (int i = 0; i < 100000000; i++) s += (seed + i) % 7;
    return s;
}

template <class Job>
void run(const char* kind, Job job, int nthread, int njob) {
    auto t0 = steady_clock::now();
    std::vector<std::thread> ts;
    for (int k = 0; k < nthread; k++)
        ts.emplace_back([=]() {          // k 番目のスレッドは k, k+n, k+2n ... 件目を担当
            for (int j = k; j < njob; j += nthread) job(j);
        });
    for (auto& t : ts) t.join();
    std::cout << kind << " " << nthread << " スレッド : "
              << duration_cast<milliseconds>(steady_clock::now() - t0).count() << " ms\n";
}

int main() {
    std::cout << "hardware_concurrency() = "
              << std::thread::hardware_concurrency() << "\n\n";
    for (int n : {1, 2, 4}) run("待つ仕事    ", wait_job, n, 4);
    std::cout << "\n";
    for (int n : {1, 2, 4}) run("計算する仕事", calc_job, n, 4);
    return 0;
}

In [ ]:
!g++ -std=c++17 -pthread ex01b.cpp -o ex01b && ./ex01b

**待つ仕事**の結果は、どのマシンでもほぼ同じになります。

```
待つ仕事     1 スレッド : 1200 ms
待つ仕事     2 スレッド :  600 ms   ← 人数分だけ速くなる
待つ仕事     4 スレッド :  300 ms   ←
```

一方、**計算する仕事**の結果は、マシンによってまったく違います。

```
Colab での典型例                     参考：計算回路を4つ積んだPCの例
1 スレッド : 1373 ms                 1 スレッド : 1300 ms
2 スレッド : 1356 ms                 2 スレッド :  660 ms  ← 半分
4 スレッド : 1364 ms                 4 スレッド :  340 ms  ← さらに半分
   ↑ 何スレッドにしても変わらない
```

**待つ仕事は、人数に比例してきれいに速くなります。**
待つのに計算回路は要らないので、コア数に関係なく、何人でも同時に待てるからです。

**計算する仕事は、「同時に計算できる数」で頭打ちになります。**
同時に計算できるのは、そのマシンにある計算回路の数までだからです。

そして Colab の「2コア」は、実は**計算回路1つ分の性能しかありません**。
**上の実行結果で、計算する仕事が2スレッドにしても1スレッドと変わらなかったのは、
このためです**。`hardware_concurrency()` は 2 と答えるのに、計算は1人分しか
速くならない ―― この数字は目安であって、当てにならないことがあります。
一方、計算回路を4つ積んだ普通のPCなら、4スレッドまでちゃんと速くなります。

> - **待つ仕事** ⇒ スレッドを増やした分だけ速くなる
> - **計算する仕事** ⇒ 「同時に計算できる数」まで。その数はマシンごとに違うので、**測って確かめる**

計算が中心の仕事を **CPUバウンド**、待ちが中心の仕事を **I/Oバウンド** と呼びます。
自分のプログラムのどこがどちらなのかを意識するのが、スレッド数を決める出発点です。

### これがパイプラインの土台

実は、動画処理の各段は、どれも**「待つ」が中心の仕事**です。
CPU から見ると、どれも「**頼んで、待つ**」の形をしています。

- **動画の読み込み** ⇒ データは SDカードやディスクから届きます。CPU は
  「次のフレームをくれ」と頼んだあと、**データが届くまでやることがありません**
- **外部ハードウェア（アクセラレータ）への処理依頼** ⇒ 計算するのは**向こう側の回路**です。
  洗濯機と同じで、スイッチを入れたら働くのは洗濯機のほう。CPU は終わるのを待つだけで、
  **その間は手が空いています**
- **画面への表示** ⇒ 描画データを画面側の装置に送ったら、**描き終わるまで待つ**だけです。
  実際に描くのは装置側の仕事です

「頼んで、待つ」仕事なら、上で見たとおり、**計算回路の少ないマシンでも
スレッドの数だけ重ねられます**。待っている間、CPU は空いているからです。
次の 1-3 で、その「重ね方」＝パイプラインを見ます。

## 1-3. ケース2 ―― 段に順序があるときはパイプライン

今度は**ケース2**です。1件の中に **Read → Infer → Show** という**順序**があります。

同じフレームの Read と Infer を同時にやることはできません。読み終わっていないデータは
処理できないからです。**では、分担する意味はないのでしょうか。**

あります。**別のフレームどうしなら重ねられる**のです。
Read 係がフレーム2を読んでいる間に、Infer 係はフレーム1を処理すればよい。

以下の図で確かめます。各段 30ms、3フレーム分です。読み方は 1-1 と同じで、

- **横が時間**（1文字 = 5ms）。帯の長さがそのまま所要時間
- **1行が1本のスレッド**
- `R11111` は「**R**ead 係が**フレーム1**を処理中」という意味

```
【1スレッドで順番に】   スレッドは1本だけ
T1    R11111I11111S11111R22222I22222S22222R33333I33333S33333
Total 270 ms   →  3フレームで 11.1 FPS
```

1本のスレッドが「フレーム1の Read → Infer → Show、次にフレーム2の…」と延々こなします。
1フレームに 90ms かかるので、3フレームで 270ms です。

```
【3スレッドで（パイプライン）】   スレッドは3本
T1    R11111R22222R33333............     ← Read 係
T2    ......I11111I22222I33333......     ← Infer 係
T3    ............S11111S22222S33333     ← Show 係
Total 150 ms   →  3フレームで 20.0 FPS
```

**下の `^^^^^^` が指している列（60〜90ms のところ）を縦に見てください。**

```
T1    R11111R22222R33333............
T2    ......I11111I22222I33333......
T3    ............S11111S22222S33333
                  ^^^^^^
                  この列では
                    T1 = フレーム3 を Read
                    T2 = フレーム2 を Infer
                    T3 = フレーム1 を Show
```

**同じ瞬間に、3つの異なるフレームが3本のスレッドで進んでいます。**
これが **パイプライン** です。工場の流れ作業と同じ形です。

1-1（ケース1）では2本の行が**ぴったり重なりました**が、ここでは段に順序があるぶん
**1段分（6文字＝30ms）ずつずれて**重なります。ずれてはいますが、同時に働いている時間は確かにあります。

### 1スレッド と 3スレッドの比較

- 1スレッド ⇒ 270ms（11.1 FPS）
- 3スレッド ⇒ 150ms（20.0 FPS）

フレーム数を増やすほど差は開きます。十分に長く流し続ければ、パイプラインは
**1フレームあたり 30ms**（一番遅い段の時間）に近づき、1スレッドの 90ms に対して**約3倍**です。

### 段の重さが違うとどうなるか

ここまでは3つの段がどれも 30ms でした。実際には**段ごとに重さが違います**。
たとえば Read=10ms、Infer=60ms、Show=10ms の場合を考えてみましょう。

```
【1スレッドで順番に】   Read=10ms, Infer=60ms, Show=10ms
T1    R1I11111111111S1R2I22222222222S2R3I33333333333S3
Total 240 ms   →  3フレームで 12.5 FPS

【3スレッドで（パイプライン）】   Read=10ms, Infer=60ms, Show=10ms
T1    R1R2R3..................................     ← Read 係：すぐ終わって手待ち
T2    ..I11111111111I22222222222I33333333333..     ← Infer 係：ずっと働きづめ
T3    ..............S1..........S2..........S3     ← Show 係：ほとんど手待ち
Total 200 ms   →  3フレームで 15.0 FPS
```

3スレッドにしても 240ms → 200ms、**1.2倍にしかなりません。**

T2（Infer 係）だけがびっしり埋まり、**T1 と T3 はほとんど `.`（手待ち）** です。
3人に分担したのに、実際にはほぼ1人しか働いていません。

このとき、いくら Read と Show を速くしても全体は変わりません。
**パイプラインの速さ（スループット）は、一番遅い段で決まります。**
この一番遅い段を **ボトルネック** と呼びます。

速くしたいなら、まず「どの段が一番遅いか」を測る。
ボトルネック以外をどれだけ速くしても、全体はほとんど変わりません。
これは並行処理でくり返し出てくる、もっとも大事な考え方の1つです。

なお、ここでもレイテンシは縮んでいません。1件あたり 10+60+10=80ms かかるのは
1スレッドでもパイプラインでも同じです。上がったのは FPS（＝スループット）だけです。

> **細かいことですが**：上の図で Read 係は最初にどんどん先へ進めてしまえます。
> 実際にこれをやると、読んだデータが行き場を失ってメモリにどんどん溜まります。
> 段と段のあいだに置く「キュー」に**容量の上限**を設けてこれを防ぎます（演習6で扱います）。

## 発展課題

1. 各段の所要時間が **Read=13ms、Infer=67ms、Show=33ms** だったとします。
   **3フレーム分**を1スレッドで順番に処理すると、何 ms かかりますか。
   3スレッドのパイプラインにすると何 ms になりますか。
   **どの段がボトルネックですか。** 十分に長く流し続けたときの上限は何 FPS でしょうか。

2. さらに、真ん中（重い処理）だけを大幅に速くして **Read=13ms、Infer=2ms、Show=33ms**
   にしたとします（これも **3フレーム分**で考えてください）。
   **ボトルネックはどこに移りますか。**
   真ん中を33倍速くしたのに、上限の FPS は何倍にしかならないでしょうか。

3. 課題2の状態で、**真ん中の段を担当する人を2人に増やした**らどうなるでしょうか。
   意味があるか、ないか、理由とともに考えてください。

4. もっと一般に、ある段の担当を2人に増やせば、その段はいつでも2倍のペースで処理できるでしょうか。
   **2人にしても倍にならない場合**を1つ挙げてみてください。

5. `ex01a.cpp` から `t1.join(); t2.join();` の2行を消すとどうなるか予測し、確かめてください。
   なぜそうなるのでしょうか。

6. `hardware_concurrency()` が **2** のマシンで、**計算しかしない仕事**（1本あたり300ms分）を
   持つスレッドを **3本** 立てたとします。

   - (a) 3本目のスレッドは、コアが空くまで**動かずに待たされる**のでしょうか。
   - (b) 3本とも終わるまでに、全部で何 ms かかるでしょうか。
     また、3本は**同時に終わる**でしょうか、**1本ずつ順に終わる**でしょうか。
   - (c) 同じことを「**ひたすら待つだけの仕事**」（300ms 待つ）でやったら、答えは変わるでしょうか。

   紙の上でタイムラインを描いてから、解答編と見比べてください。
